# PathSnap — Model: Decision Tree (CART)

**Family:** Tree-based. **Task framing:** every GPS observation has a set of candidate road arcs
(all arcs within 50 m, at most 20). Each (observation, candidate) pair is one row; the model predicts
`label = 1` if the candidate is the reference arc. The highest-probability candidate is the observation-level
prediction, and the probabilities become **emission scores** for the HMM/Viterbi stage.

**Same data, features, CV and metrics as the other PathSnap model notebooks (Logistic Regression, XGBoost, Random Forest):**
- features and labels from `Feature Extraction/extract_features.py` → `data_features/` (24 features),
- training on the 76 `use_for_tuning` train trips, rows with `label = -1` ignored,
- `GroupKFold(n_splits=5)` on `trip_id` (identical folds to the other notebooks),
- Top-1 counts observations whose reference arc has no candidate as **wrong**,
- student_test (10 trips) is used only to measure finished models — never to choose hyper-parameters.

The teammates' **Logistic Regression** and **XGBoost** configurations are re-run here through the *same* CV and
student_test evaluation, so section 6 is a like-for-like comparison.

| Section | Content |
|---|---|
| 0 | Setup and data paths |
| 1 | Load features, labels, per-observation reference |
| 2 | Metric functions |
| 3 | Why a Decision Tree; fully grown (untuned) tree with group CV |
| 4 | `max_depth` sweep (group CV) and selection |
| 5 | Overfitting analysis |
| 6 | Final fit → student_test; comparison with Logistic Regression and XGBoost |
| 7 | Classification metrics, confusion matrix, ROC, calibration |
| 8 | Tree structure and feature importance |
| 9 | Folium maps for all 10 student_test trips + static maps |
| 10 | Save model, experiment log, predictions, comparison table |

## 0. Setup

In [ ]:
# ---- Paths: local (Jupyter / VS Code) or Google Colab -------------------------------------------
import os, sys, json, time, zipfile, platform, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install folium xgboost
    from google.colab import drive
    drive.mount("/content/drive")

def unzip(z, dest):
    with zipfile.ZipFile(z) as f:
        f.extractall(dest, members=[n for n in f.namelist() if not n.startswith("__MACOSX") and not n.endswith(".DS_Store")])

def find_root():
    # ROOT must contain data_clean/ (cleaning output). data_features/ is used if present, else built.
    for c in [".", "..", "~/Downloads/PathSnap_B", "/content", "/content/drive/MyDrive/PathSnap"]:
        if (Path(c).expanduser() / "data_clean" / "trip_flags.csv").exists():
            return Path(c).expanduser()
    for z in ["data_clean.zip", "/content/drive/MyDrive/data_clean.zip"]:
        if Path(z).exists():
            unzip(z, "."); return Path(".")
    raise FileNotFoundError("Put data_clean/ (or data_clean.zip) next to this notebook")

ROOT = find_root()
CLEAN_DIR = ROOT / "data_clean"
FEAT_DIR = ROOT / "data_features"
if not (FEAT_DIR / "feature_params.json").exists():
    zf = Path("/content/drive/MyDrive/pathsnap_features.zip")
    if zf.exists():                                       # the zip the other model notebooks use
        unzip(zf, ROOT)
    else:                                                 # build it with the team's script (≈3 min)
        script = next(p for p in [ROOT / "scripts/extract_features.py", ROOT / "Feature Extraction/extract_features.py"] if p.exists())
        subprocess.run([sys.executable, str(script), "--clean", str(CLEAN_DIR), "--out", str(FEAT_DIR)], check=True)

FLAGS = CLEAN_DIR / "trip_flags.csv"
OUT = ROOT / "results" / "decision_tree"
MAP_DIR = OUT / "maps"
MAP_DIR.mkdir(parents=True, exist_ok=True)
print("clean:", CLEAN_DIR, "| features:", FEAT_DIR, "| output:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib, xgboost
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, log_loss, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report)

SEED = 42
np.random.seed(SEED)

FEATURES = json.loads((FEAT_DIR / "feature_params.json").read_text())["FEATURES"]
print(f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | {len(FEATURES)} features:\n{FEATURES}")

## 1. Load data

Same loader as the Logistic Regression / XGBoost notebooks, with two additions:
- rows with `label = -1` are **kept in memory but never used for training or scoring** — they are needed so that
  every observation gets a prediction (for route continuity and the maps);
- each observation gets its `segment_id` (from `track_clean.csv`) and raw lon/lat, for continuity and maps.

In [ ]:
flags = pd.read_csv(FLAGS, dtype={"trip_id": str})

def trips_of(part):
    t = flags[flags.partition == part]
    if part == "train":
        t = t[t.use_for_tuning.astype(bool)]
    return sorted(t.trip_id)

def load_partition(part):
    rows, pts = [], []
    for tid in trips_of(part):
        f = pd.read_csv(FEAT_DIR / part / f"{tid}.csv.gz", dtype={"trip_id": str})
        y = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}.csv.gz")
        assert len(f) == len(y) and (f.idx.values == y.idx.values).all() and (f.arc_id.values == y.arc_id.values).all()
        f["label"] = y.label.values
        f[FEATURES] = f[FEATURES].astype(np.float32)          # halves memory
        rows.append(f)
        p = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}_points.csv.gz")
        trk = pd.read_csv(CLEAN_DIR / part / tid / "track_clean.csv", usecols=["idx", "lon", "lat", "segment_id"])
        p = p.merge(trk, on="idx", how="left")
        p.insert(0, "trip_id", tid); p["part"] = part
        pts.append(p)
    return pd.concat(rows, ignore_index=True), pd.concat(pts, ignore_index=True)

t0 = time.time()
train_all, train_pts = load_partition("train")
test_all, test_pts = load_partition("student_test")
train = train_all[train_all.label >= 0].reset_index(drop=True)     # training / scoring rows
test = test_all[test_all.label >= 0].reset_index(drop=True)
del train_all
print(f"loaded in {time.time() - t0:.1f}s")
for name, d, p in [("train", train, train_pts), ("student_test", test, test_pts)]:
    L = p[p.labelled]
    print(f"{name:13} trips {d.trip_id.nunique():3} | rows {len(d):>9,} | positive rate {d.label.mean():.3f} "
          f"| labelled observations {len(L):>7,} | candidate miss rate {1 - L.ref_in_candidates.mean():.4%}")

## 2. Metrics

**Row level** (threshold 0.5, rows with `label ≥ 0`): accuracy, precision / recall / F1 of the true-arc class,
**Macro F1** (mean F1 of both classes), log loss. ~88% of rows are negatives, so row accuracy alone is misleading.

**Observation level** (the primary metrics):
- **Top-1 accuracy** — for each labelled observation, the highest-probability candidate vs the reference arc.
  Observations whose reference arc is not a candidate count as **wrong** (same as the other notebooks).
- **Observation Macro F1** — macro F1 over arc IDs (`trip:arc`) of the Top-1 predictions; rare arcs count equally.
- **Route continuity** — share of consecutive observations (same segment) whose predicted arcs are equal or
  connected within ≤ 2 arcs in the directed network.
- **Nearest-arc rule** (`dist_rank == 0`) — non-ML baseline.

Ties in probability (common for a tree: many candidates share a leaf) are broken by the row order of the feature
files, which is sorted by distance — i.e. **the closer candidate wins**.

In [ ]:
_net = {}
def network(part, tid):
    # arc endpoints + successor sets of each node (canonical arcs) for continuity
    if (part, tid) not in _net:
        z = np.load(CLEAN_DIR / part / tid / "network_clean.npz")
        c = z["arc_is_canonical"]; u, v = z["arc_u"], z["arc_v"]
        _net[(part, tid)] = (u, v, pd.Series(v[c]).groupby(u[c]).apply(set).to_dict())
    return _net[(part, tid)]

def predict_arcs(df, score, pts):
    # Highest-scoring candidate for every observation of the trips in df (-1 = no candidate).
    d = df[["trip_id", "idx", "arc_id"]].assign(s=score)
    best = d.loc[d.groupby(["trip_id", "idx"]).s.idxmax(), ["trip_id", "idx", "arc_id"]]
    P = pts[pts.trip_id.isin(df.trip_id.unique())].merge(best.rename(columns={"arc_id": "pred_arc"}),
                                                         on=["trip_id", "idx"], how="left")
    P["pred_arc"] = P.pred_arc.fillna(-1).astype(int)
    return P

def continuity(P):
    ok = tot = 0
    for (part, tid), g in P.groupby(["part", "trip_id"]):
        u, v, s = network(part, tid)
        g = g.sort_values("idx"); a = g.pred_arc.values; sg = g.segment_id.values
        for i in range(1, len(a)):
            if sg[i] != sg[i - 1] or a[i] < 0 or a[i - 1] < 0:
                continue
            tot += 1; p, q = a[i - 1], a[i]
            ok += (p == q) or (v[p] == u[q]) or (u[q] in s.get(v[p], ()))
    return ok / max(tot, 1)

def obs_metrics(P):
    L = P[P.labelled]
    hit = L.pred_arc == L.ref_arc
    return {"top1": hit.mean(), "top1_given_candidate": hit.sum() / max(L.ref_in_candidates.sum(), 1),
            "obs_macro_f1": f1_score(L.trip_id + ":" + L.ref_arc.astype(str),
                                     L.trip_id + ":" + L.pred_arc.astype(str), average="macro"),
            "n_obs": len(L)}

def evaluate(model, df, pts, with_continuity=False, nearest=False):
    # df may contain label -1 rows: they get predictions but are never scored.
    t0 = time.time(); p = model.predict_proba(df[FEATURES])[:, 1]; pred_s = time.time() - t0
    lab = df.label.values >= 0
    y, pl = df.label.values[lab], p[lab]
    yhat = (pl >= 0.5).astype(int)
    m = {"accuracy": accuracy_score(y, yhat), "precision": precision_score(y, yhat, zero_division=0),
         "recall": recall_score(y, yhat, zero_division=0), "f1": f1_score(y, yhat, zero_division=0),
         "macro_f1": f1_score(y, yhat, average="macro"),
         "log_loss": log_loss(y, np.clip(pl, 1e-7, 1 - 1e-7), labels=[0, 1])}
    P = predict_arcs(df, p, pts)
    m.update(obs_metrics(P))
    m["predict_s"] = pred_s
    m["infer_ms_per_obs"] = 1000 * pred_s / max(P.shape[0], 1)
    if with_continuity:
        m["route_continuity"] = continuity(P)
    if nearest:
        m["top1_nearest_arc"] = obs_metrics(predict_arcs(df, -df.dist_rank.values.astype(float), pts))["top1"]
    return m, p, P

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(train, train.label, groups=train.trip_id))

def cv_run(make_model, name, train_scores=True, with_continuity=False):
    # Fit on 4 folds of trips, score on the held-out trips AND on the training folds (overfitting gap).
    rows = []
    for fold, (tr, va) in enumerate(folds):
        a, b = train.iloc[tr], train.iloc[va]
        model = make_model()
        t0 = time.time(); model.fit(a[FEATURES], a.label); fit_s = time.time() - t0
        mv, _, _ = evaluate(model, b, train_pts, with_continuity=with_continuity)
        row = {"model": name, "fold": fold, "fit_s": fit_s, **{f"val_{k}": v for k, v in mv.items()}}
        if hasattr(model, "get_depth"):
            row.update({"depth": model.get_depth(), "leaves": model.get_n_leaves()})
        if train_scores:
            mt, _, _ = evaluate(model, a, train_pts)
            row.update({f"train_{k}": v for k, v in mt.items()})
        rows.append(row)
    return pd.DataFrame(rows)

## 3. Decision Tree

**Why it suits the task**
- The decision is a set of interacting if-then rules: *"closest arc **and** heading agrees"*, *"heading only counts
  when the vehicle moves"*, *"near a junction, use the neighbours' distances"*. A tree learns exactly such rules.
- No feature scaling; handles mixed features (distances in metres, cosines, counts, flags) directly.
- **Fully interpretable**: the top splits can be read and explained in the viva (section 8).
- Fast to train and extremely fast to predict — good for the API.

**Limitations:** high variance (small data changes → different tree); a fully grown tree memorises the training
trips; probabilities are **piece-wise constant leaf frequencies** — coarse and badly calibrated, which matters for HMM
emissions; axis-parallel splits approximate smooth relations (e.g. distance) with steps.

**Objective / loss:** CART — every split is chosen greedily to minimise the weighted **Gini impurity**
$G = 1 - \sum_k p_k^2$ of the two children. Leaf probability = fraction of true-arc rows in that leaf.

**Untuned baseline:** sklearn defaults — fully grown (`max_depth=None`, `min_samples_leaf=1`).

In [ ]:
t0 = time.time()
cv_default = cv_run(lambda: DecisionTreeClassifier(random_state=SEED), "DT-default", with_continuity=True)
print(f"done in {time.time() - t0:.0f}s")
cv_default[["fold", "depth", "leaves", "val_top1", "val_macro_f1", "val_log_loss", "val_route_continuity",
            "train_top1", "train_macro_f1", "train_log_loss", "fit_s"]].round(4)

## 4. `max_depth` sweep (group CV)

`max_depth ∈ {2, 3, 4, 6, 8, 10, 12, 15, 20}` with `min_samples_leaf = 20` (a leaf needs ≥ 20 rows, so its probability
is an average, not one row's label).

**Selection rule (decided before looking at results; same rule as the Random Forest notebook):** highest mean CV
**Top-1**; depths within 0.001 of the best are tie-broken by lower mean CV **log loss** (better HMM emissions).
Bayesian (Optuna) tuning is done later only for the model finally selected across all families.

In [ ]:
DEPTHS = [2, 3, 4, 6, 8, 10, 12, 15, 20]
t0 = time.time(); runs = []
for d in DEPTHS:
    r = cv_run(lambda d=d: DecisionTreeClassifier(max_depth=d, min_samples_leaf=20, random_state=SEED), f"depth={d}")
    runs.append(r.assign(max_depth=d))
    print(f"depth {d:>2} | CV top-1 {r.val_top1.mean():.4f} | macro F1 {r.val_macro_f1.mean():.4f} "
          f"| log loss {r.val_log_loss.mean():.4f} | train top-1 {r.train_top1.mean():.4f} | {time.time() - t0:.0f}s")
sweep = pd.concat(runs, ignore_index=True)
sweep.to_csv(OUT / "dt_depth_sweep_folds.csv", index=False)

sweep_s = sweep.groupby("max_depth").agg(
    cv_top1=("val_top1", "mean"), cv_top1_std=("val_top1", "std"), train_top1=("train_top1", "mean"),
    cv_macro_f1=("val_macro_f1", "mean"), cv_macro_f1_std=("val_macro_f1", "std"), train_macro_f1=("train_macro_f1", "mean"),
    cv_log_loss=("val_log_loss", "mean"), train_log_loss=("train_log_loss", "mean"), leaves=("leaves", "mean"),
    fit_s=("fit_s", "mean"))
sweep_s.round(4)

In [ ]:
best = sweep_s.cv_top1.max()
BEST_DEPTH = int(sweep_s[sweep_s.cv_top1 >= best - 0.001].cv_log_loss.idxmin())
BEST_PARAMS = dict(max_depth=BEST_DEPTH, min_samples_leaf=20, criterion="gini", random_state=SEED)
SEL = f"DT-depth{BEST_DEPTH}"
print("Selected:", SEL, BEST_PARAMS)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
s = sweep_s
ax[0].errorbar(s.index, s.cv_top1, yerr=s.cv_top1_std, marker="o", capsize=3, label="CV (held-out trips)")
ax[0].plot(s.index, s.train_top1, ls="--", marker="s", label="train folds")
ax[1].errorbar(s.index, s.cv_macro_f1, yerr=s.cv_macro_f1_std, marker="o", capsize=3, label="CV (held-out trips)")
ax[1].plot(s.index, s.train_macro_f1, ls="--", marker="s", label="train folds")
ax[2].plot(s.index, s.cv_log_loss, marker="o", label="CV (held-out trips)")
ax[2].plot(s.index, s.train_log_loss, ls="--", marker="s", label="train folds")
for a, t in zip(ax, ["Top-1 accuracy", "Row Macro F1", "Log loss"]):
    a.axvline(BEST_DEPTH, ls=":", c="gray"); a.set(xlabel="max_depth", title=f"{t}: train folds vs CV"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(OUT / "dt_validation_curves.png", dpi=140); plt.show()

## 5. Overfitting analysis

Mean over the 5 folds for the fully grown tree and the selected depth. A large positive train−CV gap (or a train
log loss far below the CV log loss) means the tree memorises its training trips. Section 7 repeats the check on the
full training set vs student_test.

In [ ]:
cv_sel = cv_run(lambda: DecisionTreeClassifier(**BEST_PARAMS), SEL, with_continuity=True)
cv_dt = pd.concat([cv_default, cv_sel], ignore_index=True)

def overfit_table(cv):
    of = cv.groupby("model", sort=False).agg(
        train_top1=("train_top1", "mean"), cv_top1=("val_top1", "mean"), cv_top1_std=("val_top1", "std"),
        train_macro_f1=("train_macro_f1", "mean"), cv_macro_f1=("val_macro_f1", "mean"), cv_macro_f1_std=("val_macro_f1", "std"),
        train_log_loss=("train_log_loss", "mean"), cv_log_loss=("val_log_loss", "mean"),
        cv_obs_macro_f1=("val_obs_macro_f1", "mean"), cv_continuity=("val_route_continuity", "mean"), fit_s=("fit_s", "mean"))
    of["top1_gap (train-CV)"] = of.train_top1 - of.cv_top1
    of["macro_f1_gap (train-CV)"] = of.train_macro_f1 - of.cv_macro_f1
    return of

of = overfit_table(cv_dt)
of["depth"] = cv_dt.groupby("model", sort=False).depth.mean()
of["leaves"] = cv_dt.groupby("model", sort=False).leaves.mean()
of.round(4).T

## 6. Comparison with the teammates' models — same CV, same student_test, same metrics

The configurations are copied from the GitHub notebooks:
- **Logistic Regression** (`Model Training/logistic_regression.ipynb`): StandardScaler + `LogisticRegression(C=1, lbfgs, max_iter=2000)`.
  Its student_test Top-1 here should reproduce that notebook's printed value (0.8616) — a check that the data are identical.
- **XGBoost** (`Model Training/xgboost_classifier.ipynb`): `binary:logistic`, hist, 300 trees, depth 10, lr 0.1,
  subsample 0.8, colsample 0.8, no early stopping.

Each is run through the same 5-fold group CV (with train-fold scores), then refit on all 76 trips and evaluated once
on student_test.

In [ ]:
def make_logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000, random_state=SEED))

def make_xgb():
    return XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", n_estimators=300,
                         max_depth=10, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, min_child_weight=1,
                         reg_lambda=1.0, scale_pos_weight=1.0, n_jobs=-1, random_state=SEED)

MAKERS = {"DT-default": lambda: DecisionTreeClassifier(random_state=SEED),
          SEL: lambda: DecisionTreeClassifier(**BEST_PARAMS),
          "Logistic Regression": make_logreg,
          "XGBoost": make_xgb}
FAMILY = {"DT-default": "Tree (CART)", SEL: "Tree (CART)", "Logistic Regression": "Linear / probabilistic",
          "XGBoost": "Boosting"}
CONFIG = {"DT-default": "fully grown, min_samples_leaf=1",
          SEL: f"max_depth={BEST_DEPTH}, min_samples_leaf=20, gini",
          "Logistic Regression": "StandardScaler + C=1.0, l2, lbfgs",
          "XGBoost": "300 trees, depth 10, lr 0.1, subsample 0.8, colsample 0.8"}

t0 = time.time()
cv_refs = []
for name in ["Logistic Regression", "XGBoost"]:
    cv_refs.append(cv_run(MAKERS[name], name, with_continuity=True))
    print(f"{name}: CV done ({time.time() - t0:.0f}s)")
cv_all = pd.concat([cv_dt] + cv_refs, ignore_index=True)
cv_all.to_csv(OUT / "cv_folds_all_models.csv", index=False)
overfit_table(cv_all).round(4).T

In [ ]:
final_models, results, preds = {}, {}, {}
for name, make in MAKERS.items():
    m = make()
    t0 = time.time(); m.fit(train[FEATURES], train.label); fit_s = time.time() - t0
    r, p, P = evaluate(m, test_all, test_pts, with_continuity=True, nearest=True)
    cvm = cv_all[cv_all.model == name]
    r.update({"fit_s": fit_s, "cv_top1": cvm.val_top1.mean(), "cv_macro_f1": cvm.val_macro_f1.mean(),
              "cv_macro_f1_std": cvm.val_macro_f1.std()})
    final_models[name], results[name], preds[name] = m, r, (p, P)
    print(f"{name:<22} fit {fit_s:6.1f}s | test top-1 {r['top1']:.4f} | macro F1 {r['macro_f1']:.4f} | log loss {r['log_loss']:.4f}")

res = pd.DataFrame(results)
res.to_csv(OUT / "student_test_results_all_models.csv")
res.loc[["cv_top1", "cv_macro_f1", "top1", "top1_given_candidate", "top1_nearest_arc", "obs_macro_f1", "macro_f1",
         "f1", "precision", "recall", "accuracy", "log_loss", "route_continuity", "fit_s", "infer_ms_per_obs"]].round(4)

### 6.1 Comparison table (report format)

In [ ]:
table = pd.DataFrame([{
    "Model": n, "Family": FAMILY[n], "Key config": CONFIG[n],
    "CV Macro F1": f"{r['cv_macro_f1']:.4f} ± {r['cv_macro_f1_std']:.4f}",
    "Top-1 Acc": round(r["top1"], 4), "Route continuity": round(r["route_continuity"], 4),
    "Student-test Macro F1": round(r["macro_f1"], 4), "Log loss": round(r["log_loss"], 4),
    "Inference time (ms/obs)": round(r["infer_ms_per_obs"], 4), "Train time (s)": round(r["fit_s"], 1)}
    for n, r in results.items()])
table.to_csv(OUT / "comparison_table.csv", index=False)
table

In [ ]:
names = list(results); colors = ["#c9b3a0", "#dd8452", "#55a868", "#8172b2"]
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
x = np.arange(len(names))
for a, key, title in [(ax[0], "cv_macro_f1", "CV Macro F1 (5-fold GroupKFold)"), (ax[1], "top1", "Student_test Top-1 accuracy")]:
    vals = [results[n][key] for n in names]
    b = a.bar(x, vals, color=colors); a.bar_label(b, fmt="%.4f", fontsize=8)
    a.set_xticks(x, names, rotation=15, fontsize=8); a.set_ylim(min(vals) - 0.05, 1.0); a.set_title(title)
ax[0].errorbar(x, [results[n]["cv_macro_f1"] for n in names], yerr=[results[n]["cv_macro_f1_std"] for n in names],
               fmt="none", c="k", capsize=4)
b = ax[2].bar(x, [results[n]["log_loss"] for n in names], color=colors); ax[2].bar_label(b, fmt="%.4f", fontsize=8)
ax[2].set_xticks(x, names, rotation=15, fontsize=8); ax[2].set_title("Student_test log loss (lower = better)")
plt.tight_layout(); plt.savefig(OUT / "model_comparison.png", dpi=140); plt.show()

### 6.2 All metrics on student_test — table and heatmaps

One row per model, every metric in one place. *Row-level* metrics use threshold 0.5 on the (observation, candidate)
rows; *observation-level* metrics (Top-1, observation Macro F1, continuity) are what map matching actually needs.
The heatmap colours each column from worst (light) to best (dark); for log loss lower is better.

In [ ]:
METRICS = {  # column name -> (key in results, higher_is_better)
    "Accuracy (rows)": ("accuracy", True), "Precision (true arc)": ("precision", True),
    "Recall (true arc)": ("recall", True), "F1 (true arc)": ("f1", True), "Macro F1 (rows)": ("macro_f1", True),
    "ROC-AUC": ("roc_auc", True), "Log loss": ("log_loss", False), "Top-1 accuracy (obs)": ("top1", True),
    "Macro F1 (obs)": ("obs_macro_f1", True), "Route continuity": ("route_continuity", True),
    "CV Top-1": ("cv_top1", True), "CV Macro F1": ("cv_macro_f1", True)}

lab_mask = test_all.label.values >= 0
for n in results:                                  # ROC-AUC from the stored student_test probabilities
    results[n]["roc_auc"] = roc_auc_score(test_all.label.values[lab_mask], preds[n][0][lab_mask])

metrics_table = pd.DataFrame({col: {n: results[n][k] for n in results} for col, (k, _) in METRICS.items()})
metrics_table.to_csv(OUT / "all_metrics_student_test.csv")

# styled table (renders as a coloured table in Jupyter / VS Code)
higher = [c for c, (_, h) in METRICS.items() if h]
display(metrics_table.style.format("{:.4f}")
        .background_gradient(cmap="Greens", subset=higher, axis=0)
        .background_gradient(cmap="Greens_r", subset=["Log loss"], axis=0)
        .set_caption("Student_test metrics — all models (darker = better in each column)"))

# heatmap image for the report: each column min-max scaled so colours compare models within a metric
scaled = metrics_table.copy()
for c, (_, h) in METRICS.items():
    v = scaled[c]; rng = v.max() - v.min()
    scaled[c] = (v - v.min()) / rng if rng > 0 else 1.0
    if not h:
        scaled[c] = 1 - scaled[c]
fig, ax = plt.subplots(figsize=(17, 1.1 + 0.6 * len(metrics_table)))
im = ax.imshow(scaled.values, cmap="Greens", aspect="auto", vmin=0, vmax=1)
for i in range(metrics_table.shape[0]):
    for j in range(metrics_table.shape[1]):
        ax.text(j, i, f"{metrics_table.iat[i, j]:.4f}", ha="center", va="center", fontsize=9,
                color="white" if scaled.iat[i, j] > 0.6 else "black")
ax.set_xticks(range(metrics_table.shape[1]), metrics_table.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(metrics_table.shape[0]), metrics_table.index, fontsize=10)
ax.set_title("Metrics heatmap — student_test (colour = rank within each metric, darker = better; values printed)")
plt.tight_layout(); plt.savefig(OUT / "metrics_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Confusion-matrix heatmaps for every model: counts (top) and row-normalised % (bottom)
names = list(results)
fig, axes = plt.subplots(2, len(names), figsize=(4.6 * len(names), 8.4))
y_rows = test_all.label.values[lab_mask]
for j, n in enumerate(names):
    yh = (preds[n][0][lab_mask] >= 0.5).astype(int)
    cm = confusion_matrix(y_rows, yh)
    for i, (mat, fmt, ttl) in enumerate([(cm, ",", "counts"), (cm / cm.sum(1, keepdims=True), ".2%", "% of true class")]):
        ax = axes[i, j]
        ax.imshow(mat, cmap="Blues")
        for r in range(2):
            for c in range(2):
                ax.text(c, r, format(mat[r, c], fmt), ha="center", va="center", fontsize=12,
                        color="white" if mat[r, c] > mat.max() / 2 else "black")
        ax.set_xticks([0, 1], ["pred: not arc", "pred: true arc"]); ax.set_yticks([0, 1], ["not arc", "true arc"])
        ax.set_title(f"{n}\n({ttl})", fontsize=10)
plt.suptitle("Confusion matrices — student_test rows (threshold 0.5)", weight="bold")
plt.tight_layout(); plt.savefig(OUT / "confusion_heatmaps_all_models.png", dpi=140); plt.show()

### 6.3 Per-trip Top-1 on student_test

In [ ]:
p_sel, P_sel = preds[SEL]
rows = []
for tid, g in P_sel.groupby("trip_id"):
    L = g[g.labelled]; hit = L.pred_arc == L.ref_arc
    row = {"trip_id": tid, "labelled_obs": len(L),
           "mean_candidates": test_all[test_all.trip_id == tid].groupby("idx").size().mean(),
           "top1": hit.mean(), "obs_macro_f1": f1_score(L.ref_arc, L.pred_arc, average="macro"),
           "wrong": int((~hit).sum()), "continuity": continuity(g)}
    for n in ["Logistic Regression", "XGBoost"]:
        Q = preds[n][1]; Q = Q[(Q.trip_id == tid) & Q.labelled]
        row[f"top1_{n}"] = (Q.pred_arc == Q.ref_arc).mean()
    rows.append(row)
per_trip = pd.DataFrame(rows).set_index("trip_id")
per_trip.to_csv(OUT / "dt_student_test_per_trip.csv")

fig, ax = plt.subplots(figsize=(14, 4)); x = np.arange(len(per_trip)); w = 0.27
for k, (col, lab, c) in enumerate([("top1_Logistic Regression", "Logistic Regression", "#55a868"),
                                   ("top1", SEL, "#dd8452"), ("top1_XGBoost", "XGBoost", "#8172b2")]):
    ax.bar(x + (k - 1) * w, per_trip[col], w, label=lab, color=c)
ax.set_xticks(x, per_trip.index); ax.set_ylim(0.5, 1.0); ax.set_ylabel("Top-1 accuracy"); ax.legend(loc="lower right")
for i, n in enumerate(per_trip.mean_candidates): ax.text(i, 0.51, f"{n:.0f} cand", ha="center", fontsize=8, color="gray")
ax.set_title("Per-trip Top-1 accuracy on student_test (classifier only)")
plt.tight_layout(); plt.savefig(OUT / "per_trip_top1.png", dpi=140); plt.show()

# per-trip heatmap: trips x models
ht = per_trip[["top1_Logistic Regression", "top1", "top1_XGBoost"]].rename(
    columns={"top1": SEL, "top1_Logistic Regression": "Logistic Regression", "top1_XGBoost": "XGBoost"})
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(ht.values, cmap="RdYlGn", vmin=max(0.5, ht.values.min()), vmax=1, aspect="auto")
for i in range(ht.shape[0]):
    for j in range(ht.shape[1]):
        ax.text(j, i, f"{ht.iat[i, j]:.3f}", ha="center", va="center", fontsize=9)
ax.set_xticks(range(ht.shape[1]), ht.columns); ax.set_yticks(range(ht.shape[0]), ht.index)
plt.colorbar(im, ax=ax, label="Top-1 accuracy"); ax.set_title("Per-trip Top-1 heatmap (student_test)")
plt.tight_layout(); plt.savefig(OUT / "per_trip_heatmap.png", dpi=140); plt.show()
per_trip.round(4)

## 7. Classification metrics of the selected tree — train vs student_test

Row level, threshold 0.5. Plain accuracy is inflated by the ~88% negative rows; quote the **true-arc F1**,
**Macro F1** and the observation-level **Top-1**. The train column is the second overfitting check.

In [ ]:
dt = final_models[SEL]

def clf_metrics(model, df, name):
    p = model.predict_proba(df[FEATURES])[:, 1]; y = df.label.values; yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh).ravel()
    return {"Dataset": name, "Rows": len(y), "Accuracy": accuracy_score(y, yh),
            "Precision (true arc)": precision_score(y, yh), "Recall (true arc)": recall_score(y, yh),
            "F1 (true arc)": f1_score(y, yh), "F1 (not true arc)": f1_score(y, yh, pos_label=0),
            "Macro F1": f1_score(y, yh, average="macro"), "Weighted F1": f1_score(y, yh, average="weighted"),
            "ROC-AUC": roc_auc_score(y, p), "Log loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}, y, p

m_tr, _, _ = clf_metrics(dt, train, "Train (76 trips)")
m_te, y_te, p_te = clf_metrics(dt, test, "Student_test (10 trips)")
perf = pd.DataFrame([m_tr, m_te]).set_index("Dataset").T
perf.to_csv(OUT / "dt_classification_metrics.csv")
print(classification_report(y_te, (p_te >= 0.5).astype(int), target_names=["not true arc (0)", "true arc (1)"], digits=4))
perf

In [ ]:
L = P_sel[P_sel.labelled]
pd.Series({"Labelled GPS observations": len(L),
           "Correctly matched": int((L.pred_arc == L.ref_arc).sum()),
           "Wrongly matched": int((L.pred_arc != L.ref_arc).sum()),
           "Top-1 matching accuracy": results[SEL]["top1"],
           "Nearest-arc baseline Top-1": results[SEL]["top1_nearest_arc"],
           "Observation Macro F1": results[SEL]["obs_macro_f1"],
           "Route continuity": results[SEL]["route_continuity"]}, name="student_test").to_frame()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
ConfusionMatrixDisplay(confusion_matrix(y_te, (p_te >= 0.5).astype(int)), display_labels=["not true arc", "true arc"]).plot(
    ax=ax[0], colorbar=False, values_format=",", cmap="Blues")
ax[0].set_title(f"{SEL}: confusion matrix (student_test rows)")
for n, c in [(SEL, "#dd8452"), ("Logistic Regression", "#55a868"), ("XGBoost", "#8172b2")]:
    p = final_models[n].predict_proba(test[FEATURES])[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p); ax[1].plot(fpr, tpr, c=c, label=f"{n} (AUC {roc_auc_score(y_te, p):.4f})")
    fr, mp = calibration_curve(y_te, p, n_bins=10, strategy="quantile"); ax[2].plot(mp, fr, marker="o", c=c, label=n)
ax[1].plot([0, 1], [0, 1], "k:", lw=1); ax[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (student_test)")
ax[1].legend(loc="lower right")
ax[2].plot([0, 1], [0, 1], "k:", lw=1, label="perfect calibration")
ax[2].set(xlabel="mean predicted probability", ylabel="observed fraction of true arcs",
          title="Calibration — are the probabilities honest?\n(they become HMM emission probabilities)")
ax[2].legend()
plt.tight_layout(); plt.savefig(OUT / "dt_confusion_roc_calibration.png", dpi=140); plt.show()

## 8. Tree structure and feature importance

The top three levels of the selected tree are the rules it relies on most; **Gini importance** is the total impurity
decrease contributed by each feature over all its splits.

In [ ]:
plt.figure(figsize=(24, 9))
plot_tree(dt, max_depth=3, feature_names=FEATURES, class_names=["not true arc", "true arc"],
          filled=True, impurity=False, proportion=True, fontsize=8)
plt.title(f"{SEL}: top 3 levels"); plt.savefig(OUT / "dt_tree_top3.png", dpi=110, bbox_inches="tight"); plt.show()

imp = pd.Series(dt.feature_importances_, index=FEATURES).sort_values()
imp.to_csv(OUT / "dt_feature_importance.csv")
fig, ax = plt.subplots(figsize=(8, 7)); imp.plot.barh(ax=ax, color="#4c72b0")
ax.set_title(f"{SEL}: Gini feature importance"); plt.tight_layout()
plt.savefig(OUT / "dt_feature_importance.png", dpi=140); plt.show()
imp.sort_values(ascending=False).round(4).to_frame("gini_importance")

## 9. Folium maps — all 10 student_test trips

Basemaps are free and need **no API key**: OpenStreetMap, CartoDB Positron, Esri World Imagery (satellite).
Overlays (toggle in the layer control, top right): road network within 80 m, raw GPS track, reference route (green),
predicted route (red, dashed), GPS observations coloured by correctness (green = correct, red = wrong,
grey = unlabelled), start / end markers. The legend box shows that trip's metrics. Click a point for details.

In [ ]:
import folium
from folium.plugins import Fullscreen, MiniMap
from scipy.spatial import cKDTree

def load_net(tid):
    return np.load(CLEAN_DIR / "student_test" / tid / "network_clean.npz")

def arc_line(z, a):
    return [(z["node_lat"][z["arc_u"][a]], z["node_lon"][z["arc_u"][a]]),
            (z["node_lat"][z["arc_v"][a]], z["node_lon"][z["arc_v"][a]])]

def chain(z, arcs):
    # merge consecutive connected arcs into polylines; a break starts a new polyline
    lines, cur = [], []
    for a in arcs:
        seg = arc_line(z, a)
        if cur and cur[-1] == seg[0]:
            cur.append(seg[1])
        else:
            if cur: lines.append(cur)
            cur = list(seg)
    if cur: lines.append(cur)
    return lines

def nearby_arcs(z, g, radius_m=80):
    k = 111320.0; c = np.cos(np.radians(g.lat.mean()))
    d, _ = cKDTree(np.c_[g.lon * k * c, g.lat * k]).query(np.c_[z["node_lon"] * k * c, z["node_lat"] * k],
                                                         distance_upper_bound=radius_m)
    near = np.isfinite(d)
    return np.flatnonzero(z["arc_is_canonical"] & (near[z["arc_u"]] | near[z["arc_v"]]))

def make_map(tid):
    z = load_net(tid); t = per_trip.loc[tid]
    g = P_sel[P_sel.trip_id == tid].sort_values("idx")
    route = pd.read_csv(CLEAN_DIR / "labels" / "student_test" / tid / "route_clean.csv")

    m = folium.Map(tiles=None, control_scale=True)
    folium.TileLayer("openstreetmap", name="OpenStreetMap").add_to(m)
    folium.TileLayer("cartodbpositron", name="CartoDB Positron").add_to(m)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Tiles &copy; Esri", name="Esri satellite").add_to(m)

    fg = folium.FeatureGroup("Road network (within 80 m)")
    for a in nearby_arcs(z, g):
        folium.PolyLine(arc_line(z, a), color="#7f7f7f", weight=1, opacity=0.55).add_to(fg)
    fg.add_to(m)

    fg = folium.FeatureGroup("Raw GPS track", show=False)
    for _, s in g.groupby("segment_id"):
        folium.PolyLine(list(zip(s.lat, s.lon)), color="#1f77b4", weight=1.5, opacity=0.7).add_to(fg)
    fg.add_to(m)

    fg = folium.FeatureGroup("Reference route")
    for line in chain(z, route.canonical_arc_id.values):
        folium.PolyLine(line, color="#2ca02c", weight=7, opacity=0.55, tooltip="reference route").add_to(fg)
    fg.add_to(m)

    pa = g.pred_arc[g.pred_arc >= 0].values
    pa = pa[np.r_[True, pa[1:] != pa[:-1]]]                      # drop consecutive duplicates
    fg = folium.FeatureGroup("Predicted route (DT top-1)")
    for line in chain(z, pa):
        folium.PolyLine(line, color="#d62728", weight=3, opacity=0.95, dash_array="6,4", tooltip="predicted").add_to(fg)
    fg.add_to(m)

    fg = folium.FeatureGroup("GPS observations (by correctness)")
    for r in g.itertuples():
        wrong = r.labelled and r.pred_arc != r.ref_arc
        col = "#9e9e9e" if not r.labelled else ("#d73027" if wrong else "#1a9850")
        folium.CircleMarker((r.lat, r.lon), radius=4.5 if wrong else 2.5, color=col, fill=True, fill_opacity=0.9, weight=1,
                            popup=f"idx {r.idx}<br>predicted arc {r.pred_arc}<br>reference arc {r.ref_arc}").add_to(fg)
    fg.add_to(m)

    folium.Marker((g.lat.iloc[0], g.lon.iloc[0]), tooltip="start", icon=folium.Icon(color="green", icon="play")).add_to(m)
    folium.Marker((g.lat.iloc[-1], g.lon.iloc[-1]), tooltip="end", icon=folium.Icon(color="black", icon="stop")).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    Fullscreen().add_to(m); MiniMap(toggle_display=True).add_to(m)

    m.get_root().html.add_child(folium.Element(f'''
    <div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:white;padding:10px 12px;
                border:1px solid #888;border-radius:6px;font:13px Arial;box-shadow:0 1px 4px rgba(0,0,0,.3)">
      <b>PathSnap · trip {tid}</b> · {SEL} (classifier only)<br>
      Top-1 <b>{t.top1:.3f}</b> · obs Macro F1 <b>{t.obs_macro_f1:.3f}</b> · continuity <b>{t.continuity:.3f}</b><br>
      {int(t.labelled_obs):,} labelled obs · {int(t.wrong):,} wrong · {t.mean_candidates:.1f} candidates/obs
      <hr style="margin:5px 0">
      <span style="color:#2ca02c;font-size:16px">&#9473;</span> reference route &nbsp;
      <span style="color:#d62728;font-size:16px">&#9477;</span> predicted route<br>
      <span style="color:#1a9850">&#9679;</span> correct &nbsp;<span style="color:#d73027">&#9679;</span> wrong &nbsp;
      <span style="color:#9e9e9e">&#9679;</span> unlabelled &nbsp;<span style="color:#7f7f7f">&#9472;</span> road network
    </div>'''))
    m.fit_bounds([[g.lat.min(), g.lon.min()], [g.lat.max(), g.lon.max()]], padding=(20, 20))
    return m

for tid in per_trip.index:
    make_map(tid).save(str(MAP_DIR / f"trip_{tid}_dt_map.html"))
print(f"saved {len(per_trip)} maps to {MAP_DIR}")
SHOW_TRIP = per_trip.top1.idxmin()             # hardest trip for the tree — change to any trip id
print("showing trip", SHOW_TRIP)
make_map(SHOW_TRIP)

### 9.1 Static maps for the PDF report (same layers, no basemap)

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

def static_map(tid, ax):
    z = load_net(tid); t = per_trip.loc[tid]
    g = P_sel[P_sel.trip_id == tid].sort_values("idx")
    route = pd.read_csv(CLEAN_DIR / "labels" / "student_test" / tid / "route_clean.csv")
    ax.add_collection(LineCollection([[(lo, la) for la, lo in arc_line(z, a)] for a in nearby_arcs(z, g)],
                                     colors="#bdbdbd", linewidths=0.5, zorder=1))
    for line in chain(z, route.canonical_arc_id.values):
        la, lo = zip(*line); ax.plot(lo, la, c="#2ca02c", lw=4, alpha=0.5, zorder=2)
    pa = g.pred_arc[g.pred_arc >= 0].values; pa = pa[np.r_[True, pa[1:] != pa[:-1]]]
    for line in chain(z, pa):
        la, lo = zip(*line); ax.plot(lo, la, c="#d62728", lw=1.2, ls="--", zorder=3)
    ok = g.labelled & (g.pred_arc == g.ref_arc); bad = g.labelled & (g.pred_arc != g.ref_arc)
    ax.scatter(g.lon[ok], g.lat[ok], s=3, c="#1a9850", zorder=4)
    ax.scatter(g.lon[bad], g.lat[bad], s=10, c="#d73027", zorder=5)
    ax.set_aspect(1 / np.cos(np.radians(g.lat.mean()))); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"Trip {tid} · Top-1 {t.top1:.3f} · cont. {t.continuity:.3f}", fontsize=9)

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, tid in zip(axes.ravel(), per_trip.index):
    static_map(tid, ax)
fig.legend(handles=[Line2D([], [], c="#2ca02c", lw=4, alpha=.5, label="reference route"),
                    Line2D([], [], c="#d62728", ls="--", label=f"predicted route ({SEL})"),
                    Line2D([], [], c="#bdbdbd", label="road network"),
                    Line2D([], [], marker="o", ls="", c="#1a9850", label="correct observation"),
                    Line2D([], [], marker="o", ls="", c="#d73027", label="wrong observation")],
           loc="lower center", ncol=5, fontsize=11)
fig.suptitle(f"PathSnap — {SEL} classifier-only route reconstruction on the 10 student_test trips", weight="bold")
plt.tight_layout(rect=(0, 0.05, 1, 0.96))
plt.savefig(OUT / "dt_static_maps_all_trips.png", dpi=150); plt.show()

## 10. Save model, experiment log, predictions and comparison table

- `dt_model.joblib` — the selected tree.
- `experiment_log.json` — configuration, seed, versions, CV folds, depth sweep, student_test results of all four models.
- `student_test_predictions.csv.gz` — per-candidate probabilities (`p_dt`) → HMM emission scores.
- `comparison_table.csv` — the report table rows (written in section 6.1).

In [ ]:
joblib.dump(dt, OUT / "dt_model.joblib")
test_all[["trip_id", "idx", "arc_id", "label"]].assign(p_dt=p_sel).to_csv(
    OUT / "student_test_predictions.csv.gz", index=False, float_format="%.6f")

log = {
    "config": {"model": "DecisionTreeClassifier", "family": "Tree (CART)", **BEST_PARAMS,
               "cv": "GroupKFold(n_splits=5) on trip_id",
               "selection_rule": "max mean CV top-1; within 0.001 -> min mean CV log loss"},
    "features": FEATURES,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "data": {"train_trips": trips_of("train"), "student_test_trips": trips_of("student_test"),
             "train_rows": len(train), "student_test_rows": len(test)},
    "cv_folds": cv_all.to_dict(orient="records"),
    "depth_sweep": sweep_s.reset_index().to_dict(orient="records"),
    "student_test": {k: {a: float(b) for a, b in v.items()} for k, v in results.items()},
    "student_test_per_trip": per_trip.reset_index().to_dict(orient="records"),
}
(OUT / "experiment_log.json").write_text(json.dumps(log, indent=2, default=float))
print("saved to", OUT, "→", sorted(p.name for p in OUT.iterdir()))

## 11. Viva notes

Quote numbers from the outputs above, not from memory.
1. **Why a tree:** the decision is a set of interacting rules (distance × heading × motion × context); a tree learns
   them directly, needs no scaling, and its top splits can be read (section 8).
2. **No test leakage:** depth chosen by group CV on the 76 train trips; student_test used once at the end.
3. **Overfitting:** the fully grown tree vs the depth-limited tree — compare train vs CV in section 5 and
   train vs student_test in section 7. `min_samples_leaf=20` also stops single-row leaves (probabilities of exactly 0/1),
   which is what blows up the fully grown tree's log loss.
4. **Same data as LR/XGBoost:** the Logistic Regression rerun here reproduces its own notebook's student_test Top-1.
5. **Probabilities matter for the HMM:** compare log loss and the calibration curve, not only Top-1.